# Chronos-2 trading features with SB3 PPO

This notebook mirrors the Chronos trading setup from the native REINFORCE tutorial, but trains `stable_baselines3.PPO`. The install surface stays compact: `crosslearn[chronos]` covers the package functionality, while `gym-anytrading`, `pandas`, and `matplotlib` remain notebook-level dependencies.
            

In [ ]:
# Uncomment in a fresh environment:
# %pip install "crosslearn[chronos]" gym-anytrading pandas matplotlib 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from gym_anytrading.datasets import STOCKS_GOOGL
from gym_anytrading.envs import StocksEnv

from stable_baselines3 import PPO
from crosslearn.extractors import ChronosExtractor, build_offline_bundle

In [ ]:
LOOKBACK = 30
FEATURE_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Volume']
SELECTED_COLUMNS = ['Close', 'Volume']
FRAME_BOUND = (LOOKBACK, len(STOCKS_GOOGL))

display(STOCKS_GOOGL.head(31))
print('Frame bound:', FRAME_BOUND)
print('Feature columns:', FEATURE_COLUMNS)
print('Chronos-selected columns:', SELECTED_COLUMNS)        

In [ ]:
def online_process_data(env):
    start = env.frame_bound[0] - env.window_size
    end = env.frame_bound[1]
    prices = env.df.loc[:, 'Close'].to_numpy()[start:end]
    signal_features = env.df.loc[:, FEATURE_COLUMNS].to_numpy(dtype=np.float32)[start:end]
    return prices, signal_features


class OnlineStocksEnv(StocksEnv):
    _process_data = online_process_data


def make_online_env():
    return OnlineStocksEnv(
        df=STOCKS_GOOGL,
        window_size=LOOKBACK,
        frame_bound=FRAME_BOUND,
    )

sample_env = make_online_env()
print('Online observation space:', sample_env.observation_space)
sample_env.close()      

## Online Chronos extraction with PPO

The extractor is supplied through SB3 `policy_kwargs`. For PPO rollout, batch, and optimization details beyond this minimal setup, use the SB3 documentation as the main reference. This notebook keeps the setup intentionally small; on GPU, larger parallel batches are still what make Chronos inference meaningfully wider.
            

In [ ]:
def render_single_episode(env, agent):
    obs, _ = env.reset()
    terminated = False
    truncated = False
    
    while not (terminated or truncated):
        action, _ = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Info: {info}")
    env.unwrapped.render_all()
    plt.show()
    env.close()

In [ ]:
env = make_online_env()
model = PPO(
    'MlpPolicy',
    env,
    n_steps=512,
    policy_kwargs={
        'features_extractor_class': ChronosExtractor,
        'features_extractor_kwargs': {
            'feature_names': FEATURE_COLUMNS,
            'selected_columns': SELECTED_COLUMNS,
        },
    },
    verbose=1,
    seed=42,
)
model.learn(total_timesteps=10_000)      

In [ ]:
render_single_episode(make_online_env(), model)

## Offline Chronos embeddings

Offline mode uses `build_offline_bundle(...)` to precompute the Chronos vectors first, then hands those vectors to PPO as plain numeric observations. Set `progress_bar=True` to monitor the offline embedding pass.
            

In [30]:
offline_bundle = build_offline_bundle(
    STOCKS_GOOGL,
    lookback=LOOKBACK,
    frame_bound=FRAME_BOUND,
    feature_columns=FEATURE_COLUMNS,
    selected_columns=SELECTED_COLUMNS,
    progress_bar=True,
)
display(offline_bundle["df"].head())
display(offline_bundle["embedding_frame"].head())
print("Offline embedding frame shape:", offline_bundle["embedding_frame"].shape)
print("Offline embedding columns:", offline_bundle["embedding_frame"].columns[:5].tolist(), "...")

Chronos embeddings:   0%|          | 0/2306 [00:00<?, ?window/s]

,Open,High,Low,Close,Adj Close,Volume,chronos_0,chronos_1,chronos_2,chronos_3,...,chronos_758,chronos_759,chronos_760,chronos_761,chronos_762,chronos_763,chronos_764,chronos_765,chronos_766,chronos_767
0,203.453461,205.525528,201.031036,205.010010,205.010010,4520600,-0.051934,-0.067057,-0.054964,-0.880732,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
1,204.324326,204.799805,198.188187,198.513519,198.513519,6512000,-0.051934,-0.067057,-0.054964,-0.880732,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
2,200.200195,203.203201,199.229233,201.446442,201.446442,6875500,-0.051934,-0.067057,-0.054964,-0.880732,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
3,203.263260,207.432434,203.103104,205.400406,205.400406,6544600,-0.051934,-0.067057,-0.054964,-0.880732,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
4,204.994995,208.893890,204.554550,207.407410,207.407410,5847300,-0.051934,-0.067057,-0.054964,-0.880732,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743


,chronos_0,chronos_1,chronos_2,chronos_3,chronos_4,chronos_5,chronos_6,chronos_7,chronos_8,chronos_9,...,chronos_758,chronos_759,chronos_760,chronos_761,chronos_762,chronos_763,chronos_764,chronos_765,chronos_766,chronos_767
0,-0.051934,-0.067057,-0.054964,-0.880732,0.264007,0.337688,1.761837,-0.172182,-0.696817,0.222838,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
1,-0.051934,-0.067057,-0.054964,-0.880732,0.264007,0.337688,1.761837,-0.172182,-0.696817,0.222838,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
2,-0.051934,-0.067057,-0.054964,-0.880732,0.264007,0.337688,1.761837,-0.172182,-0.696817,0.222837,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
3,-0.051934,-0.067057,-0.054964,-0.880732,0.264007,0.337688,1.761837,-0.172182,-0.696817,0.222838,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743
4,-0.051934,-0.067057,-0.054964,-0.880732,0.264007,0.337688,1.761837,-0.172182,-0.696817,0.222837,...,-0.134393,0.354705,0.512628,0.193808,0.224893,0.779282,0.435093,0.209851,0.116631,0.634743


Offline embedding frame shape: (2306, 768)
Offline embedding columns: ['chronos_0', 'chronos_1', 'chronos_2', 'chronos_3', 'chronos_4'] ...


In [ ]:
class OfflineStocksEnv(StocksEnv):
    def __init__(self, prices, signal_features, **kwargs):
        self._prices = prices
        self._signal_features = signal_features.astype(np.float32)
        super().__init__(**kwargs)

    def _process_data(self):
        return self._prices, self._signal_features


def make_offline_env():
    return OfflineStocksEnv(
        prices=offline_bundle["df"]["Close"].to_numpy(dtype=np.float32),
        signal_features=offline_bundle["embedding_frame"].to_numpy(dtype=np.float32),
        df=offline_bundle["df"],
        window_size=1,
        frame_bound=(1, len(offline_bundle["df"])),
    )

offline_env = make_offline_env()
offline_model = PPO("MlpPolicy", 
                    offline_env, 
                    verbose=1, 
                    seed=42)
offline_model.learn(total_timesteps=10_000)

In [ ]:
render_single_episode(make_offline_env(), offline_model)

## Evaluate the Offline-Trained Agent in the Online Chronos Environment.

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.monitor import Monitor

# Assuming you already have:
#   - offline_model (the PPO model you trained on pre-embedded observations)
#   - The same constants you used for embedding (from the notebook)

# 1.1 Define your online environment (raw rolling windows)
#    (the notebook defines OnlineStocksEnv / make_online_env for this)
# 1.2 Create a new PPO instance configured for *online* Chronos (extractor does the embedding)
policy_kwargs = {
    "features_extractor_class": ChronosExtractor,
    "features_extractor_kwargs": {
        "feature_names": FEATURE_COLUMNS,
        "selected_columns": SELECTED_COLUMNS,
        # Optional but recommended for reproducibility:
        # "pooling": "mean",          # or "last"
        # "device_map": "auto",
        # "features_dim": None,       # no extra projection layer by default
    },
}
online_env = make_online_env() # online environment with raw windows
eval_model = PPO(
    "MlpPolicy",
    online_env,   
    policy_kwargs=policy_kwargs,
    verbose=1,          # or 1 if you want logs
    # No learning needed — we only want the policy for prediction
)

# 2. Transfer the learned policy weights from the offline model
#    (extractor has no / different trainable params, so strict=False is safe)
eval_model.policy.load_state_dict(
    offline_model.policy.state_dict(),
    strict=False
)

In [ ]:
# 3. Now evaluate / run episodes in the online Chronos environment using sb3's built-in evaluation utility
online_env = Monitor(online_env)  # Wrap the online environment with Monitor for evaluation
mean_reward, std_reward = evaluate_policy(
    eval_model,
    online_env,
    n_eval_episodes=1,
    deterministic=True
)
print(f"Online evaluation — mean reward: {mean_reward:.2f} ± {std_reward:.2f}")

In [ ]:
# 3. Evaluate / render in the online Chronos environment
render_single_episode(online_env, eval_model)